# Stream ordering and link segmentation with D-inf and MFD routing

xarray-spatial provides stream network analysis (ordering and link segmentation) for three flow routing models:

- **D8**: single steepest-descent neighbor (existing `stream_order`, `stream_link`)
- **D-infinity**: continuous angle distributing flow between two neighbors (Tarboton 1997)
- **MFD**: flow fractions distributed to all downslope neighbors (Freeman/Quinn)

This notebook shows how to use the D-inf and MFD variants.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial import generate_terrain
from xrspatial import flow_direction, flow_direction_dinf, flow_direction_mfd
from xrspatial import flow_accumulation, flow_accumulation_mfd
from xrspatial import stream_order, stream_link
from xrspatial import stream_order_dinf, stream_link_dinf
from xrspatial import stream_order_mfd, stream_link_mfd

## Generate synthetic terrain

In [ ]:
W, H = 400, 400

template = xr.DataArray(np.zeros((H, W)), dims=['y', 'x'],
                        coords={'y': np.linspace(0, 1000, H),
                                'x': np.linspace(0, 1000, W)})

terrain = generate_terrain(template, x_range=(0, 1000), y_range=(0, 1000))

fig, ax = plt.subplots(figsize=(6, 5))
terrain.plot(ax=ax, cmap='terrain')
ax.set_title('Synthetic elevation')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Compute flow directions and accumulation

We compute all three routing models from the same elevation surface.

In [ ]:
# D8
fd_d8 = flow_direction(terrain)
fa_d8 = flow_accumulation(fd_d8)

# D-infinity
fd_dinf = flow_direction_dinf(terrain)

# MFD
fd_mfd = flow_direction_mfd(terrain)
fa_mfd = flow_accumulation_mfd(fd_mfd)

print(f'D8 flow dir shape:   {fd_d8.shape}')
print(f'D-inf angles shape:  {fd_dinf.shape}')
print(f'MFD fractions shape: {fd_mfd.shape}')

## Stream ordering: D8 vs D-inf vs MFD

We extract stream networks using a threshold and compare Strahler ordering across routing models.

In [ ]:
threshold = 200

# D8 stream order (uses D8 accumulation)
so_d8 = stream_order(fd_d8, fa_d8, threshold=threshold, method='strahler')

# D-inf stream order (uses D8 accumulation for thresholding)
so_dinf = stream_order_dinf(fd_dinf, fa_d8, threshold=threshold, method='strahler')

# MFD stream order (uses MFD accumulation for thresholding)
so_mfd = stream_order_mfd(fd_mfd, fa_mfd, threshold=threshold, method='strahler')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, data, title in zip(axes,
                            [so_d8, so_dinf, so_mfd],
                            ['D8', 'D-infinity', 'MFD']):
    im = ax.imshow(np.where(np.isnan(data.values), 0, data.values),
                   cmap='Blues', interpolation='nearest',
                   vmin=0, vmax=max(np.nanmax(so_d8.values),
                                    np.nanmax(so_dinf.values),
                                    np.nanmax(so_mfd.values)))
    ax.set_title(f'Strahler order ({title})')
    ax.set_aspect('equal')

fig.colorbar(im, ax=axes, shrink=0.6, label='Stream order')
plt.tight_layout()
plt.show()

## Shreve magnitude comparison

In [ ]:
sv_d8 = stream_order(fd_d8, fa_d8, threshold=threshold, method='shreve')
sv_dinf = stream_order_dinf(fd_dinf, fa_d8, threshold=threshold, method='shreve')
sv_mfd = stream_order_mfd(fd_mfd, fa_mfd, threshold=threshold, method='shreve')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, data, title in zip(axes,
                            [sv_d8, sv_dinf, sv_mfd],
                            ['D8', 'D-infinity', 'MFD']):
    vals = np.where(np.isnan(data.values), 0, data.values)
    im = ax.imshow(np.log1p(vals), cmap='viridis', interpolation='nearest')
    ax.set_title(f'Shreve magnitude ({title})')
    ax.set_aspect('equal')

fig.colorbar(im, ax=axes, shrink=0.6, label='log(1 + magnitude)')
plt.tight_layout()
plt.show()

## Stream link segmentation

In [ ]:
sl_d8 = stream_link(fd_d8, fa_d8, threshold=threshold)
sl_dinf = stream_link_dinf(fd_dinf, fa_d8, threshold=threshold)
sl_mfd = stream_link_mfd(fd_mfd, fa_mfd, threshold=threshold)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, data, title in zip(axes,
                            [sl_d8, sl_dinf, sl_mfd],
                            ['D8', 'D-infinity', 'MFD']):
    vals = data.values.copy()
    # Color by link ID mod a palette size for visibility
    vals_display = np.where(np.isnan(vals), 0,
                            np.mod(vals, 20) + 1)
    ax.imshow(vals_display, cmap='tab20', interpolation='nearest')
    ax.set_title(f'Stream links ({title})')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## Counting stream segments

Each routing model produces a slightly different stream network topology.
MFD and D-inf can produce more junction points than D8 because flow
splits across multiple neighbors.

In [ ]:
for label, data in [('D8', sl_d8), ('D-inf', sl_dinf), ('MFD', sl_mfd)]:
    n_links = len(np.unique(data.values[~np.isnan(data.values)]))
    n_stream = int(np.sum(~np.isnan(data.values)))
    print(f'{label:6s}: {n_links:4d} links, {n_stream:6d} stream cells')